# 11 — Observability & Logging

## Background

LLM systems fail in subtle ways: prompts drift, costs spike, latency degrades, and model behavior shifts — often invisibly. Traditional APM tools weren't built for token-level economics and stochastic outputs.

## What You'll Learn

- Structured prompt/response logging with full request context
- Token-level cost accounting per model and endpoint
- Latency percentile tracking (p50/p95/p99) with rolling windows
- Anomaly detection: cost spikes, latency degradation, error rate surges
- Trace propagation: correlating LLM calls across multi-step pipelines
- Dashboard-ready metrics export format

## Why This Matters

Without observability, LLM production systems are black boxes. You can't debug regressions, attribute costs to features, catch prompt injection attempts in traffic, or prove SLA compliance. Observability is what turns a prototype into an auditable service.

In [ ]:
import time, uuid, json, hashlib, statistics
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional
from collections import deque
from enum import Enum
import numpy as np

MODEL_COSTS = {
    'claude-opus-4-6':   {'input': 0.015,   'output': 0.075},
    'claude-sonnet-4-6': {'input': 0.003,   'output': 0.015},
    'claude-haiku-4-5':  {'input': 0.00025, 'output': 0.00125},
    'gpt-4o':            {'input': 0.005,   'output': 0.015},
    'gpt-4o-mini':       {'input': 0.00015, 'output': 0.0006},
}

def compute_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    rates = MODEL_COSTS.get(model, {'input': 0.001, 'output': 0.002})
    return (input_tokens * rates['input'] + output_tokens * rates['output']) / 1000.0

for m in ['claude-haiku-4-5', 'claude-sonnet-4-6', 'claude-opus-4-6']:
    cost = compute_cost(m, 1000, 500)
    print(f'  {m:<26}: ${cost:.6f} for 1K in + 500 out')


## Structured LLM Log Entry

In [ ]:
class CompletionStatus(str, Enum):
    SUCCESS    = 'success'
    ERROR      = 'error'
    TIMEOUT    = 'timeout'
    RATE_LIMIT = 'rate_limit'

@dataclass
class LLMLogEntry:
    request_id:     str
    trace_id:       str
    span_id:        str
    timestamp_utc:  float
    model:          str
    endpoint:       str
    prompt_hash:    str
    input_tokens:   int
    output_tokens:  int
    cost_usd:       float
    latency_ms:     float
    status:         CompletionStatus
    error_message:  Optional[str] = None
    feature_tag:    Optional[str] = None
    user_id:        Optional[str] = None
    extra:          Dict[str, Any] = field(default_factory=dict)

    def to_json(self) -> str:
        d = asdict(self)
        d['status'] = self.status.value
        return json.dumps(d)

    @property
    def total_tokens(self) -> int:
        return self.input_tokens + self.output_tokens

def make_entry(model, endpoint, input_tok, output_tok, latency_ms,
               status=CompletionStatus.SUCCESS, trace_id=None,
               feature_tag=None, error=None) -> LLMLogEntry:
    return LLMLogEntry(
        request_id    = str(uuid.uuid4()),
        trace_id      = trace_id or str(uuid.uuid4()),
        span_id       = str(uuid.uuid4()),
        timestamp_utc = time.time(),
        model         = model,
        endpoint      = endpoint,
        prompt_hash   = hashlib.sha256(f'{model}{input_tok}'.encode()).hexdigest()[:16],
        input_tokens  = input_tok,
        output_tokens = output_tok,
        cost_usd      = compute_cost(model, input_tok, output_tok),
        latency_ms    = latency_ms,
        status        = status,
        error_message = error,
        feature_tag   = feature_tag,
    )

sample = make_entry('claude-sonnet-4-6', '/v1/messages', 800, 350, 1234.5, feature_tag='chat')
print(json.dumps(json.loads(sample.to_json()), indent=2))


## Rolling-Window Metrics Collector

In [ ]:
@dataclass
class WindowMetrics:
    latencies:  deque = field(default_factory=lambda: deque(maxlen=1000))
    costs:      deque = field(default_factory=lambda: deque(maxlen=1000))
    tokens_in:  deque = field(default_factory=lambda: deque(maxlen=1000))
    tokens_out: deque = field(default_factory=lambda: deque(maxlen=1000))
    error_count: int  = 0
    total_count: int  = 0

    def record(self, entry: LLMLogEntry):
        self.latencies.append(entry.latency_ms)
        self.costs.append(entry.cost_usd)
        self.tokens_in.append(entry.input_tokens)
        self.tokens_out.append(entry.output_tokens)
        self.total_count += 1
        if entry.status != CompletionStatus.SUCCESS:
            self.error_count += 1

    def summary(self) -> Dict:
        if not self.latencies: return {}
        return {
            'n':              self.total_count,
            'error_rate':     self.error_count / max(1, self.total_count),
            'latency_p50':    float(np.percentile(list(self.latencies), 50)),
            'latency_p95':    float(np.percentile(list(self.latencies), 95)),
            'latency_p99':    float(np.percentile(list(self.latencies), 99)),
            'cost_total':     sum(self.costs),
            'cost_mean':      statistics.mean(self.costs),
            'tokens_in_mean': statistics.mean(self.tokens_in),
            'tokens_out_mean':statistics.mean(self.tokens_out),
        }

class LLMObservabilityCollector:
    def __init__(self):
        self._by_model:   Dict[str, WindowMetrics] = {}
        self._by_feature: Dict[str, WindowMetrics] = {}
        self._global:     WindowMetrics = WindowMetrics()
        self._log:        List[LLMLogEntry] = []

    def record(self, entry: LLMLogEntry):
        self._log.append(entry)
        self._global.record(entry)
        self._by_model.setdefault(entry.model, WindowMetrics()).record(entry)
        if entry.feature_tag:
            self._by_feature.setdefault(entry.feature_tag, WindowMetrics()).record(entry)

    def global_summary(self)  -> Dict: return self._global.summary()
    def model_summary(self)   -> Dict: return {m: w.summary() for m, w in self._by_model.items()}
    def feature_summary(self) -> Dict: return {f: w.summary() for f, w in self._by_feature.items()}

print('Collector defined.')


## Anomaly Detection

Compares a short recent window against a longer baseline to catch cost spikes, latency degradation, and error surges.

In [ ]:
@dataclass
class AnomalyAlert:
    kind:     str
    metric:   str
    baseline: float
    current:  float
    ratio:    float
    severity: str

    def __str__(self):
        return (f'[{self.severity.upper()}] {self.kind}: {self.metric} '
                f'is {self.ratio:.1f}x baseline ({self.baseline:.3f} -> {self.current:.3f})')

class AnomalyDetector:
    def __init__(self, warn_ratio: float = 1.5, crit_ratio: float = 3.0,
                 baseline_window: int = 500, current_window: int = 50):
        self.warn_ratio      = warn_ratio
        self.crit_ratio      = crit_ratio
        self.baseline_window = baseline_window
        self.current_window  = current_window

    def _check(self, baseline_vals, current_vals, metric, kind):
        if not baseline_vals or not current_vals: return None
        baseline = statistics.median(baseline_vals)
        current  = statistics.median(current_vals)
        if baseline <= 0: return None
        ratio = current / baseline
        if ratio >= self.crit_ratio:
            return AnomalyAlert(kind, metric, baseline, current, ratio, 'critical')
        if ratio >= self.warn_ratio:
            return AnomalyAlert(kind, metric, baseline, current, ratio, 'warning')
        return None

    def scan(self, collector: LLMObservabilityCollector) -> List[AnomalyAlert]:
        log = collector._log
        if len(log) < self.baseline_window + self.current_window: return []
        bl = log[-(self.baseline_window + self.current_window):-self.current_window]
        cl = log[-self.current_window:]
        alerts = []
        for metric, fn in [
            ('latency_ms',   lambda e: e.latency_ms),
            ('cost_usd',     lambda e: e.cost_usd),
            ('input_tokens', lambda e: float(e.input_tokens)),
        ]:
            a = self._check([fn(e) for e in bl], [fn(e) for e in cl], metric, 'spike')
            if a: alerts.append(a)
        return alerts

print('Anomaly detector defined.')


## Synthetic Traffic + Anomaly Demo

In [ ]:
rng = np.random.default_rng(42)
MODELS   = ['claude-sonnet-4-6', 'claude-haiku-4-5', 'claude-opus-4-6']
FEATURES = ['chat', 'summarize', 'code-assist', 'rag']
collector = LLMObservabilityCollector()
detector  = AnomalyDetector()

for i in range(600):
    model   = rng.choice(MODELS, p=[0.6, 0.3, 0.1])
    feature = rng.choice(FEATURES)
    in_tok  = int(rng.normal(600, 150).clip(50, 4000))
    out_tok = int(rng.normal(300, 100).clip(10, 2000))
    latency = float(rng.normal(1200, 300).clip(100, 8000))
    status  = CompletionStatus.SUCCESS if rng.random() > 0.02 else CompletionStatus.ERROR
    collector.record(make_entry(model, '/v1/messages', in_tok, out_tok,
                                latency, status=status, feature_tag=feature))

# Inject latency spike
for i in range(50):
    collector.record(make_entry('claude-sonnet-4-6', '/v1/messages',
                                600, 300, float(rng.normal(8000, 500)), feature_tag='chat'))

g = collector.global_summary()
print('=== Global Metrics ===')
print(f'  n={g["n"]}  error_rate={g["error_rate"]*100:.1f}%  '
      f'p50={g["latency_p50"]:.0f}ms  p95={g["latency_p95"]:.0f}ms  cost=${g["cost_total"]:.4f}')

print('\n=== Per-Model ===')
for model, m in collector.model_summary().items():
    print(f'  {model:<26} n={m["n"]:>4}  cost=${m["cost_total"]:.4f}  p95={m["latency_p95"]:.0f}ms')

print('\n=== Per-Feature ===')
for feat, m in collector.feature_summary().items():
    print(f'  {feat:<14} n={m["n"]:>4}  cost=${m["cost_total"]:.4f}')

print('\n=== Anomaly Scan ===')
alerts = detector.scan(collector)
for a in alerts: print(f'  {a}')
if not alerts: print('  No anomalies.')


## Trace Context Propagation

In [ ]:
from contextlib import contextmanager

class TraceContext:
    _current_trace_id: Optional[str] = None

    @classmethod
    @contextmanager
    def new_trace(cls):
        cls._current_trace_id = str(uuid.uuid4())
        try:
            yield cls._current_trace_id
        finally:
            cls._current_trace_id = None

    @classmethod
    def current(cls) -> Optional[str]: return cls._current_trace_id

collector2 = LLMObservabilityCollector()

def llm_call(step, model, in_tok, out_tok, latency):
    entry = make_entry(model, '/v1/messages', in_tok, out_tok, latency,
                       trace_id=TraceContext.current(), feature_tag=step)
    collector2.record(entry)
    return entry

print('RAG pipeline trace:')
with TraceContext.new_trace() as tid:
    print(f'  Trace ID: {tid}')
    e1 = llm_call('query-expand',  'claude-haiku-4-5',   200,  50, 350.0)
    e2 = llm_call('rerank-prompt', 'claude-haiku-4-5',  1800, 120, 890.0)
    e3 = llm_call('generate',      'claude-sonnet-4-6', 3200, 800, 2100.0)
    e4 = llm_call('format',        'claude-haiku-4-5',   600, 200, 420.0)

print('\nSpans:')
for e in [e1, e2, e3, e4]:
    assert e.trace_id == tid
    print(f'  [{e.feature_tag:<14}] {e.model:<26} '
          f'in={e.input_tokens:>4} out={e.output_tokens:>3} '
          f'cost=${e.cost_usd:.5f} lat={e.latency_ms:.0f}ms')

print(f'\n  Total cost: ${sum(e.cost_usd for e in [e1,e2,e3,e4]):.5f}')
print(f'  Total latency: {sum(e.latency_ms for e in [e1,e2,e3,e4]):.0f}ms')


## Key Takeaways

- Structured `LLMLogEntry` records enable cost attribution and debugging without storing raw prompts
- Rolling `WindowMetrics` gives real-time p50/p95/p99 without unbounded memory
- `AnomalyDetector` compares short current window against longer baseline — simple but catches spikes
- `TraceContext` propagates a single `trace_id` through multi-step pipelines for full call graph reconstruction
- Store prompt hashes by default; add opt-in raw logging only for debugging tiers